In [ ]:
import mysql.connector
from faker import Faker
import random

fake = Faker('pt_BR')

# Configurações de conexão MySQL
config = {
    'user': 'root',
    'password': '996523',
    'host': '127.0.0.1',
    'database': 'ecommerce'
}

In [ ]:


def populate_ecommerce():
    try:
        conn = mysql.connector.connect(**config)
        cursor = conn.cursor()

        # --- 1. POPULAR CATEGORIAS ---
        print("Inserindo categorias...")
        categorias = ['Eletrônicos', 'Livros', 'Moda', 'Casa', 'Games']
        for cat in categorias:
            cursor.execute("INSERT INTO categoria (nome_categoria) VALUES (%s)", (cat,))
        conn.commit()

        # --- 2. POPULAR PRODUTOS ---
        print("Inserindo produtos...")
        cursor.execute("SELECT idcategoria FROM categoria")
        ids_categorias = [row[0] for row in cursor.fetchall()]
        
        for _ in range(20):
            nome_prod = fake.ecommerce_name() if hasattr(fake, 'ecommerce_name') else fake.word().capitalize()
            cursor.execute("""
                INSERT INTO produto (nome, preco_venda, descricao, categoria_idcategoria) 
                VALUES (%s, %s, %s, %s)""",
                (nome_prod, round(random.uniform(10, 500), 2), fake.sentence(), random.choice(ids_categorias))
            )
        conn.commit()

        # --- 3. POPULAR CLIENTES (INTEGRIDADE PAI-FILHO) ---
        print("Inserindo clientes e especializações...")
        for _ in range(30):
            tipo = random.choice(['PF', 'PJ'])
            email = fake.unique.email()
            
            # Inserir Pai
            cursor.execute("INSERT INTO cliente (email, tipo_cliente) VALUES (%s, %s)", (email, tipo))
            id_cliente = cursor.lastrowid # Pega o ID gerado pelo AUTO_INCREMENT

            if tipo == 'PF':
                cursor.execute("""
                    INSERT INTO cliente_pf (idcliente, primeiro_nome, ultimo_nome, cpf, dt_nasc, sexo)
                    VALUES (%s, %s, %s, %s, %s, %s)""",
                    (id_cliente, fake.first_name(), fake.last_name(), fake.unique.cpf().replace('.','').replace('-',''), 
                     fake.date_of_birth(minimum_age=18), random.choice(['f', 'm']))
                )
            else:
                cursor.execute("""
                    INSERT INTO cliente_pj (idcliente, cnpj, razao_social, inscricao_estadual, nome_fantasia)
                    VALUES (%s, %s, %s, %s, %s)""",
                    (id_cliente, fake.unique.cnpj(), fake.company(), str(fake.random_number(digits=12)), fake.company_suffix())
                )
            
            # Inserir Telefone
            cursor.execute("INSERT INTO telefone (cliente_idcliente, numero, tipo) VALUES (%s, %s, %s)",
                          (id_cliente, fake.cellphone_number(), random.choice(['celular', 'comercial'])))

        conn.commit()
        print("Finalizado com sucesso!")

    except mysql.connector.Error as err:
        print(f"Erro no MySQL: {err}")
        conn.rollback()
    finally:
        if 'conn' in locals() and conn.is_connected():
            cursor.close()
            conn.close()

if __name__ == "__main__":
    populate_ecommerce()

Erro no MySQL: 1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)


UnboundLocalError: cannot access local variable 'conn' where it is not associated with a value

In [ ]:
def populate_orders_and_payments(n_pedidos=50):
    try:
        conn = mysql.connector.connect(**config)
        cursor = conn.cursor()

        # --- 1. BUSCAR IDs EXISTENTES ---
        cursor.execute("SELECT idcliente FROM cliente")
        ids_clientes = [row[0] for row in cursor.fetchall()]

        cursor.execute("SELECT idproduto, preco_venda FROM produto")
        produtos_disponiveis = cursor.fetchall() # Lista de tuplas (id, preco)

        print(f"Gerando {n_pedidos} pedidos...")

        for _ in range(n_pedidos):
            # --- 2. CRIAR O PEDIDO (PAI) ---
            id_cliente = random.choice(ids_clientes)
            frete = round(random.uniform(10, 50), 2)
            
            cursor.execute("""
                INSERT INTO pedido (cliente_idcliente, frete, status_pedido, valor_total) 
                VALUES (%s, %s, %s, %s)""",
                (id_cliente, frete, random.choice(['criado', 'finalizado']), 0) # Total começa em 0
            )
            id_pedido = cursor.lastrowid

            # --- 3. ADICIONAR ITENS AO PEDIDO (PRODUTO_PEDIDO) ---
            total_pedido = frete
            # Cada pedido terá de 1 a 4 produtos diferentes
            itens_no_pedido = random.sample(produtos_disponiveis, random.randint(1, 4))
            
            for prod_id, preco in itens_no_pedido:
                quantidade = random.randint(1, 3)
                total_pedido += (float(preco) * quantidade)
                
                cursor.execute("""
                    INSERT INTO produto_pedido (pedido_idpedido, produto_idproduto, quantidade) 
                    VALUES (%s, %s, %s)""",
                    (id_pedido, prod_id, quantidade)
                )

            # Atualizar o valor total real do pedido
            cursor.execute("UPDATE pedido SET valor_total = %s WHERE idpedido = %s", (total_pedido, id_pedido))

            # --- 4. GERAR PAGAMENTO ---
            metodo = random.choice(['cartao_credito', 'cartao_debito', 'boleto', 'pix'])
            status_pag = 'pago' if random.random() > 0.2 else 'pendente'
            
            cursor.execute("""
                INSERT INTO pagamento (pedido_idpedido, metodo_pagamento, status_pagamento) 
                VALUES (%s, %s, %s)""",
                (id_pedido, metodo, status_pag)
            )

        conn.commit()
        print("Pedidos e Pagamentos gerados com sucesso!")

    except mysql.connector.Error as err:
        print(f"Erro ao gerar pedidos: {err}")
        conn.rollback()
    finally:
        if conn.is_connected():
            cursor.close()
            conn.close()

# Executar a nova função
if __name__ == "__main__":
    # Certifique-se de que a função anterior (populate_ecommerce) já rodou!
    populate_orders_and_payments(50)

UnboundLocalError: cannot access local variable 'conn' where it is not associated with a value